### New attention mechanism

Given, dot-product selection attention: takes cosine similarity between data vectors.

Output: row-wise stochastic score matrix $Softmax_{row}(S)$

We need a replacement for always finding cosine overlap between Q-K and further S-V.

How about letting a learned func ($f_a$) figure out mapping between elements. So the embeddings dont have to closer or its spatial position is not the exlusive factor for attention.

The $f_a$ carries inductive bias for a domain. For example, in robot trajectory (obs.state+obs.delta_goal to action.delta_ee_pos) in vpg rl setting.



The inductive bias in robot trajectory is it, motion is continous and has to be physically feasible.

In [ ]:
Learn method first, then use method for attending obs

In [ ]:
import torch
from torch

In [ ]:

class Phase1Model(nn.Module):
    """mlp_k + predict + learnable temperature τ."""

    def __init__(self, obs_dim: int, d_z: int):
        super().__init__()
        self.mlp_k = nn.Sequential(
            nn.Linear(obs_dim, d_z * 2),
            nn.ReLU(),
            nn.Linear(d_z * 2, d_z),
        )
        self.predict = nn.Sequential(
            nn.Linear(d_z, d_z * 2),
            nn.ReLU(),
            nn.Linear(d_z * 2, d_z),
        )
        self.tau = nn.Parameter(torch.ones(1))

    def loss(self, obs: torch.Tensor) -> torch.Tensor:
        """
        obs: (B, T, obs_dim)
        Pairs: (obs_t, obs_{t+1}) for t in 0..T-2.
        L = mean ‖predict(mlp_k(obs_t)) − sg(mlp_k(obs_{t+1}))‖²
        """
        obs_t  = obs[:, :-1].reshape(-1, obs.shape[-1])   # (B*(T-1), obs_dim)
        obs_tp1 = obs[:, 1:].reshape(-1, obs.shape[-1])

        z_t   = self.mlp_k(obs_t)                         # (B*(T-1), d_z)
        z_tp1 = self.mlp_k(obs_tp1).detach()              # stop-gradient on target

        pred  = self.predict(z_t)                          # (B*(T-1), d_z)
        return F.mse_loss(pred, z_tp1)

In [ ]:
class Transformer(nn.Module):
    def __init__(self, cfg: CFG, scorer: Optional[nn.Module] = None):
        super().__init__()
        self.input_proj = nn.Linear(cfg.obs_dim, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.seq_len, cfg.d_model)
        self.blocks = nn.ModuleList([
            FDCATransformerBlock(
                d_model=cfg.d_model,
                n_heads=cfg.n_heads,
                obs_dim=cfg.obs_dim,
                ffn_mult=cfg.ffn_mult,
                scorer=scorer,
                causal=cfg.causal,
                dropout=cfg.dropout,
            )
            for _ in range(cfg.n_layers)
        ])
        self.ln_out = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.obs_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, obs_dim)  →  (B, T, obs_dim)"""
        B, T, _ = x.shape
        positions = torch.arange(T, device=x.device)
        h = self.input_proj(x) + self.pos_emb(positions)
        for block in self.blocks:
            h = block(h)
        return self.head(self.ln_out(h))
